In [2]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

c:\Users\aoezek\AppData\Local\miniconda3\envs\circadian\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
cgm_features_df=pd.read_excel(cgm_extracted_features_path)
metadata_df=pd.read_excel(metadata_path)
fasting_df=pd.read_excel(fasting_times_path)
df_analysis_ready = pd.read_excel(merged_cgm_meal_path)
ppgr_ready_not_tuned_df = pd.read_excel(models_data_ppgr_path)
keep_ids = [
    'AL16', 'AL03', 'AL23', 'AL32', 'AL24', 'AL69', 'AL35', 'AL07', 'AL22', 
    'AL25', 'AL15', 'AL47', 'AL04', 'AL26', 'AL36', 'AL13', 'AL38', 'AL37', 
    'AL57', 'AL17', 'AL06', 'AL14', 'AL21', 'AL40', 'AL11', 'AL46', 'AL30', 
    'AL05', 'AL53', 'AL02', 'AL09', 'AL08', 'AL20', 'AL48', 'AL01', 'AL45', 
    'AL51', 'AL33', 'AL60'
]
cgm_features_df_filtered = cgm_features_df[cgm_features_df['Participant_id'].isin(keep_ids)].copy()
metadata_df_filtered = metadata_df[metadata_df['ID'].isin(keep_ids)].copy()
df_analysis_ready_filtered = df_analysis_ready[df_analysis_ready['Participant_id'].isin(keep_ids)].copy()
ppgr_ready_not_tuned_filtered = ppgr_ready_not_tuned_df[ppgr_ready_not_tuned_df['Participant_id'].isin(keep_ids)].copy()

---
- The extremely small carb-content/calorie-content/gycemic index=0 meals are eliminated as for some calculations they make the denominator --> 0 and the value explode unrealistically
- then the ppgr averaged metrics are calculated
---


In [5]:
ppgr_ready_not_tuned_filtered = ppgr_ready_not_tuned_filtered[ppgr_ready_not_tuned_filtered['carbs_g'] > 5]
ppgr_ready_not_tuned_filtered = ppgr_ready_not_tuned_filtered[ppgr_ready_not_tuned_filtered['total_calories'] > 50]
ppgr_ready_not_tuned_filtered = ppgr_ready_not_tuned_filtered[ppgr_ready_not_tuned_filtered['glycemic_index'] > 0]

In [ ]:
df_analysis_ready_filtered = df_analysis_ready_filtered[df_analysis_ready_filtered['carbs_g'] > 5]
df_analysis_ready_filtered = df_analysis_ready_filtered[df_analysis_ready_filtered['total_calories'] > 50]
df_analysis_ready_filtered = df_analysis_ready_filtered[df_analysis_ready_filtered['glycemic_index'] > 0]
ppgr_profiles = extract_participant_ppgr_phenotypes(df_analysis_ready_filtered)
ppgr_profiles.head()

In [7]:
participant_profiles = extract_eating_habits(ppgr_ready_not_tuned_filtered)
metadata_df_filtered = metadata_df_filtered.rename(columns={'ID': 'Participant_id'})
fasting_df = fasting_df.rename(columns={'participant_id': 'Participant_id'})
habits_df = (
    participant_profiles
    .merge(cgm_features_df_filtered, on=['Participant_id'], how='left')
    .merge(metadata_df_filtered, on=['Participant_id'], how='left')
    .merge(fasting_df, on='Participant_id', how='left')
)

habits_df = habits_df.rename(columns={'age_x': 'age', 'sex_x':'gender', '>10':'gt_10', '<3.9':'lt_3.9'})
habits_df['gender_encoded'] = habits_df['gender'].map({'male': 0, 'female': 1})
habits_df = habits_df.drop(columns=['Unnamed: 0_y', 'Unnamed: 0_x', 'complete_cgm_data','completed_10days', 'completed_12days', 'PSQI_global_score', '10day_glucose_avg', '10day_glucose_SD', 'sex_y', 'age_y', 'self_reported_sleep_duration', 'gender'])
habits_df_noID = habits_df.drop(columns=['Participant_id'])
habits_df_clustering = (
    participant_profiles
    .merge(fasting_df, on='Participant_id', how='left')
)

habits_df_clustering_noID = habits_df_clustering.drop(columns=['Participant_id'])

In [ ]:
scores=inspect_k(ppgr_profiles, k_range=range(2, 6), is_pca=False)
scores

In [ ]:
clustered_results_only_ppgr = assign_clusters(ppgr_profiles, X_pca=None, k=2)
print(profile_clusters(clustered_results_only_ppgr))

In [ ]:
cluster_1=clustered_results_only_ppgr[clustered_results_only_ppgr['cluster']==0]
cluster_2=clustered_results_only_ppgr[clustered_results_only_ppgr['cluster']==1]

print('Number of participants in the first cluster:\n', len(cluster_1['Participant_id']),
      '\nNumber of participants in the second cluster:\n', len(cluster_2['Participant_id']))


In [ ]:
profiles_and_clusters = habits_df.merge(clustered_results_only_ppgr, on='Participant_id').drop(columns=['Participant_id'])
profiles_and_clusters_means = profiles_and_clusters.groupby('cluster').mean()
profiles_and_clusters_means.T.round(3)

---
#### PCA ONLY HABITS
---

In [ ]:
habits_df_clustering_noID=habits_df_clustering_noID.drop(columns=['glucose_baseline','Unnamed: 0','avg_daily_carbs','avg_daily_fat','avg_daily_protein','avg_daily_fiber','avg_daily_calories','diet_type'])
habits_df_clustering_noID.columns

In [ ]:
habits_df_clustering_noID_transformed = transform_features(habits_df_clustering_noID) 
X_pca, loadings, pca = reduce(habits_df_clustering_noID_transformed, variance_threshold=0.05, target_variance=0.8)

In [ ]:
loadings_rounded = loadings.round(3)
loadings_rounded

In [ ]:
print(pca.explained_variance_)
print(pca.explained_variance_ratio_)

In [ ]:
#table for thesis text
# summary_table = pd.DataFrame({
#     'Component': [f'PC{i+1}' for i in range(pca.n_components_)],
#     'Eigenvalue': pca.explained_variance_,
#     'Variance Explained (%)': pca.explained_variance_ratio_ * 100,
# })
# summary_table['Cumulative Variance (%)'] = summary_table['Variance Explained (%)'].cumsum()
# summary_table['Top Loading Feature'] = loadings.abs().idxmax().values

# summary_table = summary_table.round({
#     'Eigenvalue': 3,
#     'Variance Explained (%)': 1,
#     'Cumulative Variance (%)': 1
# })
# summary_table

---
#### save response cluster map for prediction anaylsis
---

In [ ]:
cluster_map = clustered_results_only_ppgr[['Participant_id', 'cluster']]
cluster_map['Participant_id'] = cluster_map['Participant_id'].str.replace('AL', 'P', regex=False)
cluster_map.to_excel('C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\cluster_assignments_ppgr_only.xlsx', index=False)